# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ladinski/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method Choice and Why

**Lane:** Refresh / Content Opportunity Scoring

**Business objective:** Rank existing web pages for editorial review using observable search-performance and content signals.

### Modeling Approach

This is primarily a ranking problem supported by supervised binary classification.

The target is `is_declining_label`, indicating whether a page is classified as experiencing declining search performance.

I will compare two models:

**Logistic Regression:** Provides a relatively interpretable starting point and allows us to test whether a simple learned combination of features improves ranking quality.

**Random Forest:** Can capture nonlinear relationships and interactions between search visibility, engagement, and content characteristics.

Both models will generate scores or estimated probabilities that can be used to rank pages for review.

### Week 4 Baseline

The baseline prioritizes pages with:

- Average search positions in the 4–10 or 11–20 buckets.
- CTR below the median of their position bucket.
- At least 100 impressions over 90 days.

Eligible pages are ranked by their impressions.

The earlier exploratory baseline achieved Precision@10 of 0.400 and Precision@50 of 0.520 on the starter dataset.

These exploratory scores are not directly comparable with held-out model performance. I will recompute the baseline on exactly the same test set used for the ML models.

### Evaluation

The primary metric will be Precision@50, supported by Precision@10 and Precision@20.

I will split data by client to avoid having pages from the same client in both training and evaluation sets.

The baseline's position-bucket CTR medians will be calculated from the training partition only.

### Interpretation and Limitations

I will compare predictive performance, inspect incorrect recommendations, and examine which features contribute most to the trained models.

More complex models will only be preferred if their performance justifies the additional complexity.

A prediction that a page is declining is not evidence that changing its content or search snippet will improve traffic.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split Design

**Validation method:** Client-holdout split using `GroupShuffleSplit`.

I will divide the dataset into approximately 75% training and 25% testing by client, rather than randomly splitting individual pages.

**Why client-holdout?**

Pages belonging to the same client may share content strategies, audience characteristics, and search performance patterns.

If pages from the same client appeared in both training and testing data, the model might exploit client-specific patterns and produce overly optimistic evaluation results.

**Leakage prevention:**

- Client IDs are used for splitting, not as predictive features.
- Missing-value imputation and feature scaling are fitted on training data only.
- The Week 4 baseline's CTR reference medians will be calculated on training data only.
- `trend_direction` and `trend_pct` are excluded from model inputs because they reveal the target.

**Evaluation:** The baseline and trained models will be evaluated on the same held-out pages using Precision@10, Precision@20, and Precision@50.

**Limitation:** Client-holdout validation tests generalization across clients, but does not independently establish future-period forecasting performance.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np
import subprocess
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit

repo_dir = Path("/content/flyrank-ml-internship")

# Clone repository if it is not already present
if not (repo_dir / "data/raw/content_refresh_anonymized.csv").exists():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/Ladinski/flyrank-ml-internship.git",
        str(repo_dir)
    ], check=True)

df = pd.read_csv(
    repo_dir / "data/raw/content_refresh_anonymized.csv"
)

# Binary classification target
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

# Candidate model features
feature_cols = [
    "impressions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update",
    "word_count"
]

# Find the client grouping column
client_candidates = [
    "client_hash_id",
    "client_id",
    "client_hash"
]

client_col = next(
    (c for c in client_candidates if c in df.columns),
    None
)

if client_col is None:
    raise ValueError(
        "No client ID column found. Check df.columns."
    )

# Grouped train/test split
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        df,
        df["is_declining_label"],
        groups=df[client_col]
    )
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

# Confirm no client overlap
train_clients = set(train_df[client_col])
test_clients = set(test_df[client_col])

assert train_clients.isdisjoint(test_clients)

print("Client column:", client_col)
print("Training pages:", len(train_df))
print("Testing pages:", len(test_df))
print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))
print("Client overlap:", len(train_clients & test_clients))
print("Training decline rate:",
      f"{train_df['is_declining_label'].mean():.2%}")
print("Testing decline rate:",
      f"{test_df['is_declining_label'].mean():.2%}")


Client column: client_id
Training pages: 22885
Testing pages: 7115
Training clients: 24
Testing clients: 8
Client overlap: 0
Training decline rate: 55.00%
Testing decline rate: 51.65%


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score

X_train = train_df[feature_cols]
X_test = test_df[feature_cols]

y_train = train_df["is_declining_label"].to_numpy()
y_test = test_df["is_declining_label"].to_numpy()

# 1. Week 4 baseline — fit CTR reference medians on training only
bins = [0, 3, 10, 20, 50, np.inf]
labels = ["1-3", "4-10", "11-20", "21-50", "50+"]

train_bucket = pd.cut(
    train_df["avg_position"],
    bins=bins, labels=labels, include_lowest=True
)
test_bucket = pd.cut(
    test_df["avg_position"],
    bins=bins, labels=labels, include_lowest=True
)

ctr_medians = train_df.groupby(
    train_bucket, observed=True
)["ctr"].median()

expected_ctr = test_bucket.map(ctr_medians).astype(float)

eligible = (
    test_bucket.isin(["4-10", "11-20"])
    & (test_df["ctr"] < expected_ctr)
    & (test_df["impressions_90d"] >= 100)
)

baseline_scores = np.where(
    eligible.to_numpy(),
    test_df["impressions_90d"].to_numpy(),
    0.0
)

# 2. Logistic Regression
log_model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=2000, random_state=42)
)
log_model.fit(X_train, y_train)
log_scores = log_model.predict_proba(X_test)[:, 1]

# 3. Random Forest
rf_model = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestClassifier(
        n_estimators=200,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )
)
rf_model.fit(X_train, y_train)
rf_scores = rf_model.predict_proba(X_test)[:, 1]

# Evaluate every approach on the same test labels
def precision_at_k(scores, y, k):
    order = np.argsort(-np.asarray(scores), kind="stable")
    return np.asarray(y)[order[:k]].mean()

results = []

for name, scores in [
    ("Week 4 baseline", baseline_scores),
    ("Logistic Regression", log_scores),
    ("Random Forest", rf_scores)
]:
    results.append({
        "Method": name,
        "Precision@10": precision_at_k(scores, y_test, 10),
        "Precision@20": precision_at_k(scores, y_test, 20),
        "Precision@50": precision_at_k(scores, y_test, 50)
    })

results_df = pd.DataFrame(results)
display(results_df.round(3))


,Method,Precision@10,Precision@20,Precision@50
0,Week 4 baseline,0.5,0.50,0.48
1,Logistic Regression,0.7,0.65,0.64
2,Random Forest,0.6,0.65,0.70


## 4. Errors and Interpretation

### Model Comparison

Three approaches were evaluated on the same 7,115-page client-holdout test set.

| Method | Precision@10 | Precision@20 | Precision@50 |
|---|---:|---:|---:|
| Week 4 Baseline | 0.500 | 0.500 | 0.480 |
| Logistic Regression | 0.700 | 0.650 | 0.640 |
| Random Forest | 0.600 | 0.650 | 0.700 |

Random Forest achieved the highest Precision@50, improving by 22 percentage points over the Week 4 baseline.

Logistic Regression achieved the highest Precision@10, suggesting that a simpler model may remain competitive when the review budget is especially small.

### Feature Interpretation

Permutation importance was calculated on the held-out dataset using average precision as the scoring metric.

The highest positive importance belonged to `impressions_90d` (+0.0690), followed by `avg_position` (+0.0263), `content_age_days` (+0.0189), and `ctr` (+0.0154).

`days_since_last_update` (−0.0079) and `word_count` (−0.0100) had negative permutation importance in this evaluation.

These results describe predictive contributions within this fitted model and dataset. They do not establish causal relationships.

### Error Analysis

Among the Random Forest's 50 highest-ranked pages:

- 35 were labeled as declining.
- 15 were not labeled as declining.
- Precision@50 was 0.700.

The highest-ranked page received a predicted decline probability of approximately 0.984, despite not being labeled as declining.

This demonstrates that highly confident model predictions can still be incorrect.

Some false positives also occurred among pages with low CTR or relatively poor search positions, suggesting that these signals alone cannot reliably distinguish declining from non-declining pages.

### Limitations

The current evaluation uses client-holdout validation, which separates clients between training and testing.

However, it does not independently evaluate future-period generalization.

The target represents an observed decline category, not the causal benefit of refreshing content.

Predicted probabilities have not been calibrated, and the ranking metrics were measured on a single held-out split.

### Conclusion

Random Forest is the leading candidate for this lane based on Precision@50.

However, Logistic Regression remains competitive and more interpretable.

Further work should examine performance across different client splits, probability calibration, temporal validation, and whether the model's recommendations lead to useful editorial outcomes.

Model complexity alone is not a reason to select a model.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.inspection import permutation_importance

# Inspect which features affect Random Forest performance
importance = permutation_importance(
    rf_model,
    X_test,
    y_test,
    scoring="average_precision",
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": importance.importances_mean,
    "Std": importance.importances_std
}).sort_values("Importance", ascending=False)

print("Random Forest permutation importance:")
display(importance_df.round(4))

# Inspect top-50 prediction errors
order = np.argsort(-rf_scores, kind="stable")[:50]

top50_review = test_df.iloc[order][
    ["impressions_90d", "avg_position", "ctr",
     "days_since_last_update", "is_declining_label"]
].copy()

top50_review["predicted_risk"] = rf_scores[order]
top50_review["correct_recommendation"] = (
    top50_review["is_declining_label"] == 1
)

print("\nTop-50 declining pages:",
      top50_review["correct_recommendation"].sum())

print("Top-50 incorrect recommendations:",
      (~top50_review["correct_recommendation"]).sum())

display(top50_review.head(10))


Random Forest permutation importance:


,Feature,Importance,Std
0,impressions_90d,0.0690,0.0030
1,avg_position,0.0263,0.0037
3,content_age_days,0.0189,0.0046
2,ctr,0.0154,0.0027
4,days_since_last_update,-0.0079,0.0021
5,word_count,-0.0100,0.0034



Top-50 declining pages: 35
Top-50 incorrect recommendations: 15


,impressions_90d,avg_position,ctr,days_since_last_update,is_declining_label,predicted_risk,correct_recommendation
22526,3445,39.0,0.09,104,0,0.984309,False
8652,807,24.0,0.25,104,1,0.960270,True
7367,2663,6.5,0.08,102,1,0.959381,True
14343,1828,5.6,0.22,104,1,0.956755,True
22730,1290,12.4,0.00,104,1,0.951658,True
2357,209,20.0,0.00,104,0,0.951658,False
310,836,6.8,0.24,104,1,0.951236,True
12246,1531,22.0,0.00,14,1,0.950972,True
29401,1596,6.2,0.31,104,0,0.949147,False
2150,2004,39.1,0.00,104,0,0.947960,False


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.